# Day 2:  <font color = "red">**Structure → Property Challenge**</font>

Can local image structure predict the local EELS response?

For each location, we have:

**Image patch → EELS spectrum → Scalarizer**

Your task is to:

1. Build a <font color = "yellow">**descriptor**</font> for the image patch.
2. Use a <font color = "yellow">**regressor**</font> to predict a scalarizer or the full spectrum.
3. Compare methods by:
   - predictive power
   - complexity
   - explainability

<font color = "green">**Goal**</font>

> Find the simplest explainable descriptor that predicts the EELS response well.

## 1. Explore the Dataset

Each spatial location contains:

- one image intensity value
- one full EELS spectrum

The same position in the image corresponds to the same position in the EELS spectrum image.

Let's look at one dataset.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kbarakati/athena_camm_hackathon/blob/k4my4r/docs/day_17_18092026/notebook.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
import os

In [ ]:
# !gdown --fuzzy 1FKmlRTRDImQS91V4YQjcQHoJU19X6P_6

In [ ]:
file_name = "/content/Plasmonic_sets_hackathon.npy"

raw = np.load(
    file_name,
    allow_pickle=True
)

loadedfile = raw.tolist()
print("keys:", loadedfile.keys())

In [ ]:
# Choose dataset
data = loadedfile["2"]

image = data["image"]
spectra = data["spectrum image"]
energy = data["energy axis"]

H, W = image.shape

print(image.shape)
print(spectra.shape)
print(energy.shape)

In [ ]:
# Example locations
positions = [
    (H // 4, W // 4),
    (H // 2, W // 2),
    (3 * H // 4, W // 4),
    (3 * H // 4, 3 * W // 4),
]

colors = ["C0", "C1", "C2", "C3"]

# Integrated EELS map
eels_map = spectra.sum(axis=2)

plt.rcParams.update({"font.size": 14})

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# Structural image
ax[0].imshow(image, cmap="gray")
ax[0].set_title("Structural Image")

# Spectra
for i, ((y, x), color) in enumerate(zip(positions, colors)):
    
    ax[0].scatter(x, y, s=80, color=color)
    ax[0].text(x + 3, y, str(i + 2), color=color, fontsize=18)

    ax[1].plot(
        energy,
        spectra[y, x],
        color=color,
        label=f"Point {i + 1}"
    )

    ax[2].scatter(x, y, s=80, color=color)
    ax[2].text(x + 3, y, str(i + 2), color=color, fontsize=16)

ax[1].set_title("EELS Spectra")
ax[1].set_xlabel("Energy")
ax[1].set_ylabel("Intensity")
ax[1].legend()

# EELS map
ax[2].imshow(eels_map)
ax[2].set_title("Integrated EELS Map")

plt.tight_layout()
plt.show()

## 2. Define a Scalarizer

Each EELS spectrum can be reduced to a **single target value** called a scalarizer.

For example, the spectral intensity within an energy window:

$$
S(x,y)=\int_{E_1}^{E_2} I(x,y,E)\,dE
$$

where \(E_1\) and \(E_2\) define the spectral region of interest.

Other possible scalarizers include:

- peak intensity
- peak position
- peak area
- peak ratio

## 3. Choose a Spectral Region

Before defining the scalarizer, inspect the EELS spectrum and choose an energy range containing a feature of interest.

In [ ]:
# Mean spectrum over the whole image
mean_spectrum = spectra.mean(axis=(0, 1))

plt.figure(figsize=(7, 4))

plt.plot(energy, mean_spectrum)

plt.xlabel("Energy")
plt.ylabel("EELS Intensity")
plt.title("Mean EELS Spectrum")

plt.show()

## 4. Calculate the Scalarizer

We integrate the EELS intensity inside the selected energy window to obtain one target value at each spatial location.

In [ ]:
# Choose an energy range
E1 = 0.5
E2 = 1.0

mask = (energy >= E1) & (energy <= E2)

scalarizer_map = np.trapezoid(
    spectra[:, :, mask],
    energy[mask],
    axis=2
)

plt.figure(figsize=(5, 4))
plt.imshow(scalarizer_map)
plt.colorbar(label="Integrated spectral intensity")
plt.title(f"Scalarizer Map: {E1}–{E2}")
plt.show()

## 5. Create Patch–Spectrum–Scalarizer Pairs

For each location we collect:

**Image patch → EELS spectrum → Scalarizer**

The spectrum and scalarizer correspond to the center pixel of the patch.

In [ ]:
PATCH_SIZE = 5
half = PATCH_SIZE // 2

patches = []
targets = []

for y in range(half, H - half):
    for x in range(half, W - half):

        patch = image[y-half:y+half+1, x-half:x+half+1]

        patches.append(patch)
        targets.append(scalarizer_map[y, x])

patches = np.array(patches)
targets = np.array(targets)

print("Patches:", patches.shape)
print("Targets:", targets.shape)

## 6. Look at Example Pairs

Each patch has one target scalarizer value.

Can you see structural differences between low and high target values?

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(10, 3))

ids = np.linspace(0, len(patches) - 1, 4, dtype=int)

for i, idx in enumerate(ids):
    ax[i].imshow(patches[idx], cmap="gray")
    ax[i].set_title(f"Target = {targets[idx]:.2f}")
    ax[i].axis("off")

plt.tight_layout()
plt.show()

## The Challenge

You will design two parts of the prediction pipeline:

**Image Patch → Descriptor → Regressor → EELS Scalarizer**

### Your goal

Find a combination with:

- high predictive power
- low complexity
- good explainability

You can change:

1. the **descriptor** — how the image patch is represented
2. the **regressor** — how the descriptor is mapped to the EELS property

Try different combinations and compare their performance.

## 8. Define Your Descriptor and Regressor

Change only the code inside the marked sections.

The descriptor converts an image patch into features.

The regressor learns how those features relate to the EELS scalarizer.

In [ ]:
def descriptor(patch):

    # ============================================================
    # YOUR CODE STARTS HERE
    # ============================================================

    features = [
        patch.mean(),
        patch.std(),
        patch.max() - patch.min()
    ]

    # ============================================================
    # YOUR CODE ENDS HERE
    # ============================================================

    return features


def regressor():

    # ============================================================
    # YOUR CODE STARTS HERE
    # ============================================================

    model = LinearRegression()

    # ============================================================
    # YOUR CODE ENDS HERE
    # ============================================================

    return model

## 9. Run Your Pipeline

Now we test your descriptor–regressor combination.

The same train/test split is used each time so different combinations can be compared fairly.

In [ ]:
# Build descriptor matrix
X = np.array([descriptor(patch) for patch in patches])

# Same split for every experiment
X_train, X_test, y_train, y_test = train_test_split(
    X,
    targets,
    test_size=0.25,
    random_state=0
)

# Train
model = regressor()
model.fit(X_train, y_train)

# Predict
predictions = model.predict(X_test)

# Evaluate
r2 = r2_score(y_test, predictions)

print("Regressor:", type(model).__name__)
print("Number of features:", X.shape[1])
print(f"R²: {r2:.3f}")

## 10. Compare Prediction with Ground Truth

A good model should predict values close to the measured EELS scalarizer.

Points closer to the diagonal line indicate better agreement.

In [ ]:
# Size of valid prediction region
map_shape = (
    H - 2 * half,
    W - 2 * half
)

# Predicted values
predicted = model.predict(X).reshape(map_shape)

# Put predictions back into an image of the original size
prediction_map = np.full((H, W), np.nan)

prediction_map[
    half:H-half,
    half:W-half
] = predicted


fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].imshow(scalarizer_map)
ax[0].set_title("Ground Truth")

ax[1].imshow(prediction_map)
ax[1].set_title("Prediction")

# Test data
ax[2].scatter(y_test, predictions, alpha=0.5)

low = min(y_test.min(), predictions.min())
high = max(y_test.max(), predictions.max())

ax[2].plot([low, high], [low, high], "--")
ax[2].set_xlabel("Ground Truth")
ax[2].set_ylabel("Prediction")
ax[2].set_title(f"Test Data — R² = {r2:.3f}")

plt.tight_layout()
plt.show()

## 11. Compare Descriptor–Regressor Combinations

Test several descriptors and regressors.

Create a heatmap where:

- rows = descriptors
- columns = regressors
- color/value = test R²

Use the same train/test split for every combination.

### <font color = "red">**Goal**</font>

Identify which descriptor–regressor combinations give the strongest predictive performance.

In [ ]:
# ============================================================
# YOUR CODE STARTS HERE
# ============================================================


# ============================================================
# YOUR CODE ENDS HERE
# ============================================================

## 12. Best Model and Explainability

From the heatmap, identify the best descriptor–regressor combination.

Report:

- best descriptor
- best regressor
- number of descriptor features
- test R²

Then explain:

- What does the descriptor measure?
- Which structural features are important?
- Why might those features be related to the EELS response?

In [ ]:
# ============================================================
# YOUR CODE STARTS HERE
# ============================================================


# ============================================================
# YOUR CODE ENDS HERE
# ============================================================

### Final question

> Which descriptor–regressor combination gives the best predictive performance while remaining simple and explainable?